### Imports

In [1]:
import pandas as pd
import time
from pytrends.request import TrendReq

### Loop through Wiki dataframe's game list and get Google Trend data

In [3]:
pytrends = TrendReq(
    hl='en-US',
    tz=360,
    retries=10,
    backoff_factor=0.5,
)

cache_df = pd.read_csv('data/data_sources_bronze/trends_20260527.csv')
wiki_df = pd.read_csv('data/data_sources_bronze/awards_dataset_20260516.csv')

#trends API would bomb out after so many executions and did not have enough data
#for every game. that's why we have to check and make sure it hasn't been cached already.
#Google seems to block automated pulls too, so it was a bit difficult to pull
kw_list = wiki_df['game_name'].tolist()
kw_list = [i for i in kw_list if i not in set(cache_df.columns)]

#loops through list of trends and gets game trends.
trends = dict()
for i in kw_list:
    try:
        #cat 920 is for "Board Games"
        pytrends.build_payload([i], timeframe='all', cat = 920)
        #pytrends.interest_over_time() returns a dataframe with a value from 0-100 for each month
        #a value of 0 indicates not enough data and 100 indicates peak popularity for that specific search term
        trends[i] = pytrends.interest_over_time()[i]
        time.sleep(5)
    except:
        trends_df = pd.DataFrame(trends)

### Add in the new data chunks to our cache and save it

In [7]:
if len(trends_df) > 0:
    #combine dfs, move year, and drop unnamed columns
    trends_df = trends_df.groupby(trends_df.index.year).sum()
    trends_df.index = trends_df.index.astype(int)
    cache_df.index = cache_df.index.astype(int)
    joined_df = trends_df.merge(
        cache_df,
        left_index=True,
        right_on="year",
        how="outer"
    )
    joined_df.insert(0, 'year', joined_df.pop('year'))
    joined_df = joined_df.drop(columns=[col for col in joined_df.columns if 'unnamed:' in col.lower() or 'unnamed' in col.lower()])
    joined_df = joined_df.drop(columns=[col for col in joined_df.columns if 'Unnamed:' in col or 'unnamed' in col])
    joined_df

    #i summed all of the interest over the year to get a total of interest for each product and year.
    #since our wikipedia data isn't granular enough to find which month an award was assigned to a product
    joined_df.to_csv("data/data_sources_bronze/trends_20260527.csv")

/tmp/ipykernel_618/2857723046.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  joined_df.insert(0, 'year', joined_df.pop('year'))
